# Historical Series

Optional technical review notebook. The production CLI workflow lives in `../scripts/`.


# Notebook 04 | Finalize the historical SPAM to FABLE dataset

This notebook brings the five historical years into one checked release candidate. It follows the original 2015 processing design and uses the validated year-2000 correction from Notebook 03.

**What this notebook does**

1. Verify the completed Notebook 03 audit and the exact historical candidate that passed validation.
2. Reproduce the 2015 midpoint at native 5 arc-minute grid and FABLE product resolution using the frozen 2015 country crosswalk.
3. Compare the replay with the existing 2015 spatial output and the country-level 2015 Calculator values.
4. Check the 2000 MIRCA irrigation-share correction and all five years of the Calculator table.
5. Create a versioned publication candidate with QC reports, a data dictionary, provenance hashes and a release manifest.

**What this notebook does not do**

It does not rebuild the frozen 2015 country crosswalk from raw geographic boundaries, reconstruct 2000 again, replace original production files, or approve the package for public release. A full fresh run of Notebooks 01 to 04 remains a separate reproducibility test.

**Methodological boundaries**

- Observed SPAM years: 2000, 2005, 2010, 2020.
- 2015 is the midpoint of mapped 2010 and 2020 primary area components at grid and product level. The country is assigned from the frozen 2015 crosswalk after interpolation.
- Irrigated and rainfed harvested and physical areas are processed separately. Totals and harvesting intensities are derived afterwards.
- A missing sparse endpoint row is treated as zero only within the previously validated coverage, representability and mapping conventions of the frozen production series.
- Multiple cropping is respected. Harvesting intensity is not capped at 1.
- Historical 0/0 intensity may be stored as 0 or missing. Both are checked explicitly, not converted.
- The year-2000 FABLE product completion follows the frozen evidence-based reconstruction documented in Notebook 03, including accepted FAOSTAT and MIRCA2000 magnitude support where appropriate. This is separate from irrigation-share harmonization.
- The 23 approved MIRCA2000 irrigation-share changes are carried forward from Notebook 03. They do not change harvested or physical area totals.

Run the cells from top to bottom. All outputs are written to a new publication-candidate folder.

## 01 | Project configuration and inputs

The local project location is the only setting that usually needs changing. For a second machine, set the `SPAM_FABLE_ROOT` environment variable to the folder containing `01_native_5arcmin`, `02_mapping`, `03_observed_FABLE_5arcmin` and `04_final_release`.

Use a new `RELEASE_VERSION` for a revised candidate. Existing versioned outputs are not silently overwritten. The `pythongeo` environment should include pandas, NumPy, DuckDB, PyArrow and IPython. Dependencies are not installed automatically within a publication notebook.

In [ ]:
# 01 | Environment, source paths and release settings

from pathlib import Path
import hashlib
import json
import os
import shutil
import tempfile

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)

DEFAULT_ROOT = Path(r"DATA_ROOT\SPAM_FABLE_FINAL")
ROOT = Path(os.environ.get("SPAM_FABLE_ROOT", str(DEFAULT_ROOT))).expanduser()
RELEASE_VERSION = "v1_0_candidate"

OBSERVED_DIR = ROOT / "03_observed_FABLE_5arcmin"
FINAL_DIR = ROOT / "04_final_release"
AUDIT_DIR = ROOT / "03_reconstruction_audit"
RELEASE_DIR = FINAL_DIR / "publication_candidate" / RELEASE_VERSION
QC_DIR = RELEASE_DIR / "qc"
REPLAY_DIR = RELEASE_DIR / "spatial_replay"

YEARS = [2000, 2005, 2010, 2015, 2020]
PRIMARY = [
    "harvested_irrigated_ha", "harvested_rainfed_ha",
    "physical_irrigated_ha", "physical_rainfed_ha",
]
AREAS = [
    "harvested_irrigated_ha", "harvested_rainfed_ha", "harvested_total_ha",
    "physical_irrigated_ha", "physical_rainfed_ha", "physical_total_ha",
]
KEYS = ["country_iso3", "year", "fable_product"]

UNIT_COLUMNS = {
    "harvested_irrigated_ha": "irrigated_harvested_area_1000ha",
    "harvested_rainfed_ha": "rainfed_harvested_area_1000ha",
    "harvested_total_ha": "total_harvested_area_1000ha",
    "physical_irrigated_ha": "irrigated_physical_area_1000ha",
    "physical_rainfed_ha": "rainfed_physical_area_1000ha",
    "physical_total_ha": "total_physical_area_1000ha",
}

CALCULATOR_COLUMNS = [
    *KEYS, *AREAS,
    "harvesting_intensity_irrigated", "harvesting_intensity_rainfed",
    "harvesting_intensity_total",
    "intensity_status_irrigated", "intensity_status_rainfed",
    "intensity_status_total",
    "data_status", "temporal_status", "gapfill_method", "source_dataset",
    "source_country_coverage", "fable_area_available", "mapping_complete",
    "calculator_ready", "high_intensity_review_irrigated",
    "high_intensity_review_rainfed", "high_intensity_review_total",
    "high_intensity_review_any", "intensity_quality_flag",
    *UNIT_COLUMNS.values(),
]

GRID_TOL_HA = 1e-7
COUNTRY_TOL_HA = 1e-3
AREA_TOL_HA = 1e-5
INTENSITY_TOL = 1e-10

SOURCES = {
    "products": ROOT / "02_mapping" / "02_FABLE_56_product_reference.csv",
    "observed_2010": OBSERVED_DIR / "03_SPAM_FABLE_observed_5arcmin_2010.parquet",
    "observed_2020": OBSERVED_DIR / "03_SPAM_FABLE_observed_5arcmin_2020.parquet",
    "crosswalk_2015": FINAL_DIR / "qc" / "04_grid_to_country_2015_FINAL.parquet",
    "locked_spatial_2015": FINAL_DIR / "spatial_5arcmin" / "04_SPAM_FABLE_5arcmin_2015_interpolated.parquet",
    "historical_candidate": AUDIT_DIR / "10B_mirca_historical_integration"
        / "10B_FABLE_C_SPAM_historical_2000_2020_MIRCA23_RESTORED_CANDIDATE.csv.gz",
    "mirca_changes": AUDIT_DIR / "10B_mirca_historical_integration"
        / "10B_approved_MIRCA23_changes.csv",
    "summary_10b": AUDIT_DIR / "10B_mirca_historical_integration"
        / "10B_integration_summary.csv",
    "manifest_10b": AUDIT_DIR / "10B_mirca_historical_integration"
        / "10B_release_candidate_manifest.json",
    "summary_10c": AUDIT_DIR / "10C_candidate_validation"
        / "10C_disk_validation_summary.csv",
    "manifest_10c": AUDIT_DIR / "10C_candidate_validation"
        / "10C_validation_manifest.json",
    "manifest_03": AUDIT_DIR / "03_lineage_manifest.json",
    "canonical_final": FINAL_DIR / "calculator"
        / "04_FABLE_C_SPAM_historical_input_2000_2020_FINAL.csv.gz",
}

def digest(path):
    checksum = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            checksum.update(chunk)
    return checksum.hexdigest()

def sqlpath(path):
    return Path(path).as_posix().replace("'", "''")

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as stream:
        return json.load(stream)

def require(condition, message):
    if not condition:
        raise AssertionError(message)

missing = {name: str(path) for name, path in SOURCES.items() if not path.is_file()}
if missing:
    raise FileNotFoundError(f"Required frozen inputs are missing: {missing}")

print("Project root:", ROOT)
print("Release version:", RELEASE_VERSION)
print("Release directory:", RELEASE_DIR)
print("Inputs found:", len(SOURCES), "/", len(SOURCES))

## 02 | Confirm Notebook 03 provenance

The 2000 corrections must already have passed all eight required Notebook 03 scientific gates. The historical candidate, its saved QC reports and the unchanged canonical FINAL must agree by content hash. This check prevents a release being built from a different file than the one we validated.

In [ ]:
# 02 | Cross-check the completed Notebook 03 audit

manifest_03 = load_json(SOURCES["manifest_03"])
manifest_10b = load_json(SOURCES["manifest_10b"])
manifest_10c = load_json(SOURCES["manifest_10c"])
summary_10b = pd.read_csv(SOURCES["summary_10b"])
summary_10c = pd.read_csv(SOURCES["summary_10c"])

require(len(summary_10b) == len(summary_10c) == 1, "Expected one saved QC summary per cell.")
require(manifest_03.get("audit_validated") is True, "Notebook 03 audit was not validated.")
require(manifest_03.get("release_approved") is False, "Review the changed release approval status.")
require(manifest_03.get("mandatory_audit_gates_passed") == 8, "Notebook 03 must have eight gates passed.")
require(manifest_03.get("mandatory_audit_gates_total") == 8, "Unexpected Notebook 03 gate count.")
require(
    summary_10b.loc[0, "status"] == "PASS_MIRCA23_HISTORICAL_CANDIDATE",
    "Notebook 03 historical integration is not validated.",
)
require(
    summary_10c.loc[0, "status"] == "PASS_HISTORICAL_CANDIDATE_DISK_VALIDATION",
    "Notebook 03 disk validation is not a confirmed PASS.",
)

for field in (
    "failed_mirca_area_comparisons", "new_formula_failures",
    "corrected_record_formula_failures", "baseline_formula_failures",
):
    require(int(summary_10c.loc[0, field]) == 0, f"Notebook 03 failed check: {field}")

candidate_sha = digest(SOURCES["historical_candidate"])
canonical_sha = digest(SOURCES["canonical_final"])
require(candidate_sha == manifest_10b["candidate"]["sha256"], "Historical candidate differs from Cell 10B.")
require(candidate_sha == manifest_10c["files"]["candidate"]["sha256"], "Historical candidate differs from Cell 10C.")
require(candidate_sha == manifest_03["historical_candidate_sha256"], "Historical candidate differs from Cell 11.")
require(canonical_sha == manifest_03["canonical_final_sha256"], "Canonical FINAL changed after Cell 11.")
require(canonical_sha == manifest_10b["sources"]["original_final"]["sha256"], "Canonical FINAL changed after Cell 10B.")
require(canonical_sha == manifest_10c["files"]["original_final"]["sha256"], "Canonical FINAL changed after Cell 10C.")

require(int(summary_10c.loc[0, "rows_checked"]) == 52864, "Historical row count is not 52,864.")
require(int(summary_10c.loc[0, "columns_checked"]) == 34, "Historical schema is not 34 columns.")
require(int(summary_10c.loc[0, "approved_mirca_records_matching"]) == 23, "Approved MIRCA records are incomplete.")

source_inventory = pd.DataFrame([
    {"asset": name, "path": str(path), "sha256": digest(path)}
    for name, path in SOURCES.items()
])

print("Notebook 03 audit: PASS, 8 of 8 required gates")
print("Historical candidate SHA-256:", candidate_sha)
print("Canonical FINAL SHA-256:", canonical_sha)
print("Canonical source will not be written by this notebook.")

## 03 | Check the frozen Calculator series

Keep the existing column order and country, year and product keys. We do not recalculate 2000 from current SPAM grid outputs because the validated SPAM2000 scientific completion and MIRCA share adjustments are already recorded in Notebook 03.

In [ ]:
# 03 | Load and verify the full five-year Calculator table

historical = pd.read_csv(SOURCES["historical_candidate"], low_memory=False)
reference_products = pd.read_csv(SOURCES["products"], low_memory=False)

require(list(historical.columns) == CALCULATOR_COLUMNS, "Frozen Calculator column order changed.")
require(len(historical) == 52864, "Expected exactly 52,864 historical records.")
require("fable_product" in reference_products, "Canonical product reference is missing fable_product.")

historical["country_iso3"] = historical["country_iso3"].astype("string").str.strip().str.upper()
historical["fable_product"] = historical["fable_product"].astype("string").str.strip().str.lower()
historical["year"] = pd.to_numeric(historical["year"], errors="raise").astype(int)

products = set(reference_products["fable_product"].astype("string").str.strip().str.lower().dropna())
require(len(products) == 56, "Reference must contain 56 distinct FABLE products.")
require(set(historical["year"]) == set(YEARS), "Historical years are inconsistent.")
require(not historical[KEYS].isna().any().any(), "Null historical identifiers were found.")
require(not historical.duplicated(KEYS).any(), "Duplicate country-year-product keys were found.")

for year, year_rows in historical.groupby("year"):
    require(set(year_rows["fable_product"]) == products, f"Product set differs in {year}.")
    n = year_rows.groupby("country_iso3")["fable_product"].nunique()
    require(n.eq(56).all(), f"Incomplete 56-product country panels in {year}.")

year_summary = (
    historical.groupby("year", as_index=False)
    .agg(countries=("country_iso3", "nunique"), products=("fable_product", "nunique"),
         records=("fable_product", "size"))
    .sort_values("year")
)
require(int(year_summary.loc[year_summary.year.eq(2000), "records"].iloc[0]) == 9912, "Year-2000 panel must have 9,912 records.")
require(int(year_summary.loc[year_summary.year.ne(2000), "records"].sum()) == 42952, "Non-2000 panel must have 42,952 records.")

print("Historical candidate shape:", historical.shape)
display(year_summary)

## 04 | Verify the original 2015 interpolation inputs

The original workflow interpolates by `grid_id_5min` and `fable_product`, not by country. Country borders are fixed afterwards using the previously audited 2015 country crosswalk. We verify the source keys and frozen crosswalk before running the replay.

In [ ]:
# 04 | Check the mapped anchor grids and 2015 country crosswalk

con = duckdb.connect()
con.execute("SET threads TO 4")

required_anchor_columns = {
    "grid_id_5min", "fable_product", "country_iso3", *PRIMARY,
}

for year in (2010, 2020):
    path = SOURCES[f"observed_{year}"]
    cols = set(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{sqlpath(path)}')").df()["column_name"])
    require(required_anchor_columns.issubset(cols), f"Missing mapped grid columns for {year}.")
    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW anchor_{year} AS
        SELECT grid_id_5min, LOWER(TRIM(fable_product)) AS fable_product,
               {', '.join('CAST(' + c + ' AS DOUBLE) AS ' + c for c in PRIMARY)}
        FROM read_parquet('{sqlpath(path)}')
    """)

    duplicate_count = con.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT grid_id_5min, fable_product FROM anchor_{year}
            GROUP BY 1, 2 HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    product_count = con.execute(f"SELECT COUNT(DISTINCT fable_product) FROM anchor_{year}").fetchone()[0]
    null_keys = con.execute(f"""
        SELECT COUNT(*) FROM anchor_{year}
        WHERE grid_id_5min IS NULL OR fable_product IS NULL
    """).fetchone()[0]
    require(duplicate_count == 0 and null_keys == 0, f"Invalid native grid keys for {year}.")
    require(product_count == 56, f"Expected 56 mapped products in {year}.")

crosswalk_cols = set(con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{sqlpath(SOURCES['crosswalk_2015'])}')"
).df()["column_name"])
require({"grid_id_5min", "country_iso3_2015_final"}.issubset(crosswalk_cols), "Frozen 2015 crosswalk schema changed.")

con.execute(f"""
    CREATE OR REPLACE TEMP VIEW country_2015 AS
    SELECT grid_id_5min, UPPER(TRIM(country_iso3_2015_final)) AS country_iso3
    FROM read_parquet('{sqlpath(SOURCES['crosswalk_2015'])}')
""")

crosswalk_duplicates = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT grid_id_5min FROM country_2015
        GROUP BY 1 HAVING COUNT(*) > 1
    )
""").fetchone()[0]

crosswalk_missing = con.execute("""
    WITH needed AS (
        SELECT DISTINCT grid_id_5min FROM anchor_2010
        UNION SELECT DISTINCT grid_id_5min FROM anchor_2020
    )
    SELECT COUNT(*) FROM needed n LEFT JOIN country_2015 c USING (grid_id_5min)
    WHERE c.grid_id_5min IS NULL OR c.country_iso3 IS NULL
""").fetchone()[0]

require(crosswalk_duplicates == 0, "2015 country crosswalk has duplicate grid cells.")
require(crosswalk_missing == 0, f"Unassigned anchor grids in frozen 2015 geography: {crosswalk_missing}")

print("2010 and 2020 mapped anchors: unique grid-product keys and 56 products")
print("Frozen 2015 crosswalk: unique and complete for both mapped anchor grids")

## 05 | Reproduce 2015 at native grid resolution

Each of the four primary area components is a 50:50 midpoint of the mapped 2010 and 2020 grid-product values. An absent sparse endpoint row contributes zero, as in the original validated production branch. The joined grid then receives its fixed 2015 country identifier. The replay is saved separately from the original spatial dataset.

In [ ]:
# 05 | Build the 2015 grid-product midpoint replay

QC_DIR.mkdir(parents=True, exist_ok=True)
REPLAY_DIR.mkdir(parents=True, exist_ok=True)
REPLAY_FILE = REPLAY_DIR / "04_2015_native_grid_component_midpoint_REPLAY.parquet"

components_sql = ",\n".join(
    f"(COALESCE(a.{col}, 0.0) + COALESCE(b.{col}, 0.0)) / 2.0 AS {col}"
    for col in PRIMARY
)

con.execute(f"""
    CREATE OR REPLACE TEMP VIEW midpoint_2015 AS
    WITH grid_midpoint AS (
        SELECT COALESCE(a.grid_id_5min, b.grid_id_5min) AS grid_id_5min,
               COALESCE(a.fable_product, b.fable_product) AS fable_product,
               {components_sql},
               (a.grid_id_5min IS NOT NULL) AS endpoint_2010_present,
               (b.grid_id_5min IS NOT NULL) AS endpoint_2020_present
        FROM anchor_2010 a
        FULL OUTER JOIN anchor_2020 b
          ON a.grid_id_5min = b.grid_id_5min
         AND a.fable_product = b.fable_product
    )
    SELECT m.grid_id_5min, m.fable_product, c.country_iso3,
           2015 AS year, {', '.join('m.' + name for name in PRIMARY)},
           m.harvested_irrigated_ha + m.harvested_rainfed_ha AS harvested_total_ha,
           m.physical_irrigated_ha + m.physical_rainfed_ha AS physical_total_ha,
           m.endpoint_2010_present, m.endpoint_2020_present
    FROM grid_midpoint m
    LEFT JOIN country_2015 c USING (grid_id_5min)
""")

def compare_spatial_files(left, right, tolerance=GRID_TOL_HA):
    """Compare grid-product keys, country assignment and primary area values."""
    a, b = sqlpath(left), sqlpath(right)
    difference = " OR ".join(
        f"x.{c} IS DISTINCT FROM y.{c} AND "
        f"(x.{c} IS NULL OR y.{c} IS NULL OR ABS(x.{c} - y.{c}) > {tolerance})"
        for c in PRIMARY
    )
    max_difference = ", ".join(
        f"MAX(ABS(x.{c} - y.{c})) AS max_{c}" for c in PRIMARY
    )
    return con.execute(f"""
        SELECT COUNT(*) AS compared_keys,
               SUM(CASE WHEN x.grid_id_5min IS NULL OR y.grid_id_5min IS NULL
                        THEN 1 ELSE 0 END) AS missing_keys,
               SUM(CASE WHEN x.country_iso3 IS DISTINCT FROM y.country_iso3
                        THEN 1 ELSE 0 END) AS country_mismatches,
               SUM(CASE WHEN {difference} THEN 1 ELSE 0 END) AS component_mismatches,
               {max_difference}
        FROM (
            SELECT grid_id_5min, LOWER(TRIM(fable_product)) AS fable_product,
                   UPPER(TRIM(country_iso3)) AS country_iso3,
                   {', '.join(PRIMARY)}
            FROM read_parquet('{a}')
        ) x
        FULL OUTER JOIN (
            SELECT grid_id_5min, LOWER(TRIM(fable_product)) AS fable_product,
                   UPPER(TRIM(country_iso3)) AS country_iso3,
                   {', '.join(PRIMARY)}
            FROM read_parquet('{b}')
        ) y USING (grid_id_5min, fable_product)
    """).df()

# Build in an isolated temporary folder. If a candidate replay already
# exists, verify that the fresh calculation is identical before keeping it.
with tempfile.TemporaryDirectory(prefix="04_replay_", dir=REPLAY_DIR) as temp_folder:
    new_replay = Path(temp_folder) / "2015_replayed.parquet"
    con.execute(f"""
        COPY (SELECT * FROM midpoint_2015)
        TO '{sqlpath(new_replay)}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    if REPLAY_FILE.is_file():
        existing_replay_qa = compare_spatial_files(new_replay, REPLAY_FILE)
        require(
            int(existing_replay_qa.loc[0, "missing_keys"]) == 0
            and int(existing_replay_qa.loc[0, "country_mismatches"]) == 0
            and int(existing_replay_qa.loc[0, "component_mismatches"]) == 0,
            "Existing versioned replay differs from this fresh calculation. Change RELEASE_VERSION.",
        )
    else:
        shutil.copy2(new_replay, REPLAY_FILE)

print("Independent 2015 grid midpoint replay:", REPLAY_FILE)

## 06 | Verify the locked 2015 spatial output

The replay must reproduce the locked spatial product values and the 2015 country assignment. All four primary area components are checked at individual grid and product keys. No country aggregation or intensity averaging is used to construct the replay.

In [ ]:
# 06 | Compare replay with original frozen 2015 spatial data

spatial_2015_qc = compare_spatial_files(REPLAY_FILE, SOURCES["locked_spatial_2015"])

for field in ("missing_keys", "country_mismatches", "component_mismatches"):
    require(int(spatial_2015_qc.loc[0, field]) == 0, f"2015 spatial replay mismatch: {field}")

require(int(spatial_2015_qc.loc[0, "compared_keys"]) > 0, "2015 replay has no grid cells.")
spatial_2015_qc["status"] = "PASS_2015_NATIVE_GRID_COMPONENT_REPLAY"

display(spatial_2015_qc)

## 07 | Reconcile 2015 country totals

After grid-level interpolation and frozen country assignment, aggregate the four area components by country and FABLE product. These values must agree with the 2015 records already present in the validated Calculator candidate. A country-product with no sparse spatial row may have a supported zero, but an entire country with missing grid coverage is not treated as zero.

In [ ]:
# 07 | Compare national 2015 areas to the Calculator candidate

hist_2015 = historical.loc[historical.year.eq(2015), KEYS + PRIMARY].copy()

replay_country_2015 = con.execute(f"""
    SELECT UPPER(TRIM(country_iso3)) AS country_iso3,
           fable_product, {', '.join('SUM(' + c + ') AS ' + c for c in PRIMARY)}
    FROM read_parquet('{sqlpath(REPLAY_FILE)}')
    GROUP BY 1, 2
""").df()

covered_countries = set(replay_country_2015["country_iso3"])
expected_countries = set(hist_2015["country_iso3"])
require(covered_countries == expected_countries,
        f"2015 country coverage differs: only_spatial={sorted(covered_countries - expected_countries)}, "
        f"only_calculator={sorted(expected_countries - covered_countries)}")

country_2015_qc = hist_2015.merge(
    replay_country_2015, on=["country_iso3", "fable_product"],
    how="outer", suffixes=("_calculator", "_spatial"),
    indicator=True, validate="one_to_one",
)

require(not country_2015_qc["_merge"].eq("right_only").any(),
        "A 2015 spatial product is missing from the Calculator candidate.")

for col in PRIMARY:
    expected = pd.to_numeric(country_2015_qc[f"{col}_spatial"], errors="coerce")
    actual = pd.to_numeric(country_2015_qc[f"{col}_calculator"], errors="coerce")
    missing_product = country_2015_qc["_merge"].eq("left_only")
    # Only products missing from the sparse layer receive an implicit zero.
    expected = expected.mask(missing_product, 0.0)
    country_2015_qc[f"{col}_difference_ha"] = actual - expected
    country_2015_qc[f"{col}_pass"] = (
        actual.notna() & expected.notna()
        & (actual - expected).abs().le(COUNTRY_TOL_HA)
    )

area_pass_columns = [f"{col}_pass" for col in PRIMARY]
country_2015_qc["all_components_pass"] = country_2015_qc[area_pass_columns].all(axis=1)
country_2015_failures = country_2015_qc.loc[~country_2015_qc.all_components_pass].copy()

country_2015_summary = pd.DataFrame([{
    "records_compared": len(country_2015_qc),
    "countries": len(covered_countries),
    "products": 56,
    "mismatch_records": len(country_2015_failures),
    "max_absolute_difference_ha": float(country_2015_qc[
        [f"{c}_difference_ha" for c in PRIMARY]
    ].abs().max().max()),
    "status": "PASS_2015_COUNTRY_CALCULATOR_RECONCILIATION" if country_2015_failures.empty
              else "FAIL_2015_COUNTRY_CALCULATOR_RECONCILIATION",
}])

country_2015_qc.to_csv(QC_DIR / "04_2015_country_product_comparison.csv.gz", index=False, compression="gzip")
country_2015_summary.to_csv(QC_DIR / "04_2015_country_product_summary.csv", index=False)

display(country_2015_summary)
require(country_2015_failures.empty, "Saved 2015 country totals do not match the mapped spatial midpoint. Review exported QC.")

## 08 | Check the approved 2000 correction

The validated 2000 totals are retained, while the 23 approved country-product irrigation shares carry the MIRCA2000 correction. This check compares the published sidecar with the candidate values and confirms that harvested and physical totals were not changed by the share correction.

In [ ]:
# 08 | Confirm the 23 approved MIRCA share changes

changes = pd.read_csv(SOURCES["mirca_changes"], low_memory=False)
require(len(changes) == 23, "Expected exactly 23 approved MIRCA changes.")
require(set(KEYS).issubset(changes.columns), "MIRCA sidecar is missing release keys.")

for frame in (changes,):
    frame["country_iso3"] = frame["country_iso3"].astype("string").str.strip().str.upper()
    frame["fable_product"] = frame["fable_product"].astype("string").str.strip().str.lower()
    frame["year"] = pd.to_numeric(frame["year"], errors="raise").astype(int)

require(changes.year.eq(2000).all(), "The approved MIRCA changes must apply only to year 2000.")
require(not changes.duplicated(KEYS).any(), "Duplicate approved MIRCA correction keys.")

correction_check = changes.merge(
    historical[KEYS + AREAS], on=KEYS, how="left", validate="one_to_one",
    suffixes=("_sidecar", "_candidate"), indicator=True,
)
require(correction_check["_merge"].eq("both").all(), "Some approved MIRCA keys are missing from the candidate.")

mirca_issues = {}
for col in AREAS:
    saved = pd.to_numeric(correction_check[col], errors="raise").to_numpy(float)
    approved = pd.to_numeric(correction_check[f"after_{col}"], errors="raise").to_numpy(float)
    mirca_issues[f"approved_{col}"] = int((~np.isclose(saved, approved, atol=AREA_TOL_HA, rtol=0)).sum())

for total in ("harvested_total_ha", "physical_total_ha"):
    before = pd.to_numeric(correction_check[f"before_{total}"], errors="raise").to_numpy(float)
    after = pd.to_numeric(correction_check[f"after_{total}"], errors="raise").to_numpy(float)
    mirca_issues[f"unchanged_{total}"] = int((~np.isclose(before, after, atol=AREA_TOL_HA, rtol=0)).sum())

require(sum(mirca_issues.values()) == 0, f"Approved MIRCA correction differs: {mirca_issues}")

mirca_summary = pd.DataFrame([{
    "approved_country_product_records": 23,
    "approved_area_failures": sum(v for k, v in mirca_issues.items() if k.startswith("approved_")),
    "total_area_change_failures": sum(v for k, v in mirca_issues.items() if k.startswith("unchanged_")),
    "status": "PASS_2000_MIRCA23_CORRECTION_IN_RELEASE_CANDIDATE",
}])
display(mirca_summary)

## 09 | Final five-year numerical and schema QC

This is an independent release check, not a replacement for Notebook 03's more detailed provenance audit. The 0/0 convention is checked exactly as it exists in the historical output: zero and missing intensities are both accepted only when the harvested and physical areas are both zero.

In [ ]:
# 09 | Final Calculator checks for all 52,864 historical rows

def numeric(series):
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)

def bool_array(series):
    if pd.api.types.is_bool_dtype(series):
        result = series.astype("boolean")
    else:
        result = series.astype("string").str.strip().str.lower().map({
            "true": True, "false": False, "1": True, "0": False,
            "yes": True, "no": False,
        }).astype("boolean")
    require(result.notna().all(), f"Undefined boolean values: {series.name}")
    return result.to_numpy(dtype=bool)

def allowed_intensity(h, p, saved):
    expected = np.full(h.shape, np.nan, dtype=float)
    defined = np.isfinite(h) & np.isfinite(p) & (p > 0)
    zero_zero = np.isfinite(h) & np.isfinite(p) & (h == 0) & (p == 0)
    expected[defined] = h[defined] / p[defined]
    expected[zero_zero] = 0.0
    correct = np.isclose(saved, expected, atol=INTENSITY_TOL, rtol=1e-12, equal_nan=True)
    return correct | (zero_zero & np.isnan(saved))

qc_rows = []
for area_group in ("harvested", "physical"):
    left = numeric(historical[f"{area_group}_irrigated_ha"])
    right = numeric(historical[f"{area_group}_rainfed_ha"])
    total = numeric(historical[f"{area_group}_total_ha"])
    valid = np.isfinite(left) & np.isfinite(right) & np.isfinite(total)
    qc_rows.extend([
        {"check": f"{area_group}_missing_components", "failures": int((~valid).sum())},
        {"check": f"{area_group}_negative_area", "failures": int((np.column_stack([left, right, total]) < -AREA_TOL_HA).any(axis=1).sum())},
        {"check": f"{area_group}_sum_identity", "failures": int((valid & (np.abs(left + right - total) > AREA_TOL_HA)).sum())},
    ])

for part in ("irrigated", "rainfed", "total"):
    h = numeric(historical[f"harvested_{part}_ha"])
    p = numeric(historical[f"physical_{part}_ha"])
    saved = numeric(historical[f"harvesting_intensity_{part}"])
    valid = allowed_intensity(h, p, saved)
    review = bool_array(historical[f"high_intensity_review_{part}"])
    qc_rows.extend([
        {"check": f"intensity_{part}", "failures": int((~valid).sum())},
        {"check": f"review_flag_{part}", "failures": int((review != (saved >= 5)).sum())},
    ])

any_review = bool_array(historical["high_intensity_review_any"])
parts_review = np.column_stack([
    bool_array(historical[f"high_intensity_review_{part}"])
    for part in ("irrigated", "rainfed", "total")
])
qc_rows.append({"check": "review_flag_any", "failures": int((any_review != parts_review.any(axis=1)).sum())})

for ha_col, kilo_col in UNIT_COLUMNS.items():
    h = numeric(historical[ha_col])
    k = numeric(historical[kilo_col])
    matching = np.isclose(h, k * 1000, atol=AREA_TOL_HA, rtol=0, equal_nan=True)
    qc_rows.append({"check": f"units_{kilo_col}", "failures": int((~matching).sum())})

require(bool_array(historical["calculator_ready"]).all(), "Non-ready rows in Calculator release.")
require(bool_array(historical["source_country_coverage"]).all(), "Out-of-coverage rows in Calculator release.")

numerical_qc = pd.DataFrame(qc_rows)
numerical_qc["status"] = np.where(numerical_qc.failures.eq(0), "PASS", "FAIL")
display(numerical_qc)
require(numerical_qc.failures.sum() == 0, "Final Calculator numerical QC has failures.")

final_validation = pd.DataFrame([{
    "gate": "NOTEBOOK04_RELEASE_CANDIDATE_VALIDATION",
    "historical_records": len(historical),
    "calculator_columns": len(historical.columns),
    "years": len(YEARS),
    "products": len(products),
    "mirca_corrections": 23,
    "grid_2015_mismatches": int(spatial_2015_qc.loc[0, "component_mismatches"]),
    "country_2015_mismatches": len(country_2015_failures),
    "numerical_qc_failures": int(numerical_qc.failures.sum()),
    "status": "PASS_RELEASE_CANDIDATE_SCIENTIFIC_VALIDATION",
}])
display(final_validation)

## 10 | Package the versioned publication candidate

A validated candidate is useful to share for review, but it is not the canonical production FINAL. We copy the exact compressed historical file that passed Notebook 03 validation, keep the 23-key correction sidecar, save the independent 2015 QC and document the fields and limitations.

In [ ]:
# 10 | Export final review files without touching canonical FINAL

RELEASE_FILE = RELEASE_DIR / (
    "04_FABLE_C_SPAM_historical_2000_2020_MIRCA23_CORRECTED_"
    + RELEASE_VERSION + ".csv.gz"
)
PARQUET_FILE = RELEASE_DIR / (
    "04_FABLE_C_SPAM_historical_2000_2020_" + RELEASE_VERSION + ".parquet"
)

RELEASE_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

if RELEASE_FILE.is_file():
    require(digest(RELEASE_FILE) == candidate_sha,
            "An existing release file has different content. Change RELEASE_VERSION.")
else:
    with tempfile.TemporaryDirectory(prefix="04_csv_", dir=RELEASE_DIR) as folder:
        temp_file = Path(folder) / "historical.csv.gz"
        shutil.copy2(SOURCES["historical_candidate"], temp_file)
        require(digest(temp_file) == candidate_sha, "Release copy SHA-256 mismatch.")
        shutil.copy2(temp_file, RELEASE_FILE)

# Parquet is an additional convenience format. CSV remains the original
# Calculator-facing candidate and retains its exact validated bytes.
if PARQUET_FILE.is_file():
    prior = pd.read_parquet(PARQUET_FILE)
    require(list(prior.columns) == CALCULATOR_COLUMNS and len(prior) == len(historical),
            "An existing Parquet release has a different schema or row count.")
    for col in CALCULATOR_COLUMNS:
        old, new = prior[col], historical[col]
        if pd.api.types.is_numeric_dtype(old) and pd.api.types.is_numeric_dtype(new):
            require(np.isclose(numeric(old), numeric(new), atol=1e-7, rtol=0, equal_nan=True).all(),
                    f"Existing Parquet values differ: {col}")
        else:
            require(old.astype("string").fillna("<NA>").reset_index(drop=True).equals(
                new.astype("string").fillna("<NA>").reset_index(drop=True)),
                f"Existing Parquet labels differ: {col}")
else:
    historical.to_parquet(PARQUET_FILE, index=False)

# The data dictionary covers the complete 34-column Calculator schema.
descriptions = {
    "country_iso3": "ISO 3166-1 alpha-3 country code used by the FABLE Calculator",
    "year": "Historical reference year, with 2015 interpolated",
    "fable_product": "Canonical FABLE agricultural product identifier",
    "harvested_irrigated_ha": "Irrigated harvested area in hectares",
    "harvested_rainfed_ha": "Rainfed harvested area in hectares",
    "harvested_total_ha": "Total harvested area in hectares",
    "physical_irrigated_ha": "Irrigated physical area in hectares",
    "physical_rainfed_ha": "Rainfed physical area in hectares",
    "physical_total_ha": "Total physical area in hectares",
    "harvesting_intensity_irrigated": "Irrigated harvested area divided by irrigated physical area",
    "harvesting_intensity_rainfed": "Rainfed harvested area divided by rainfed physical area",
    "harvesting_intensity_total": "Total harvested area divided by total physical area",
    "intensity_status_irrigated": "Interpretation of the irrigated intensity value",
    "intensity_status_rainfed": "Interpretation of the rainfed intensity value",
    "intensity_status_total": "Interpretation of the total intensity value",
    "data_status": "Source completeness and materialized-zero status",
    "temporal_status": "Observed SPAM or SPAM-based 2015 interpolation",
    "gapfill_method": "Year-specific time interpolation or observed-year method",
    "source_dataset": "Source dataset provenance from the frozen Calculator table",
    "source_country_coverage": "True for countries covered by the year-specific source",
    "fable_area_available": "Product representability in the source and mapping",
    "mapping_complete": "Coverage of positive source parents by accepted mapping",
    "calculator_ready": "Record passed the original Calculator export conditions",
    "high_intensity_review_irrigated": "Flag for irrigated harvesting intensity of at least 5",
    "high_intensity_review_rainfed": "Flag for rainfed harvesting intensity of at least 5",
    "high_intensity_review_total": "Flag for total harvesting intensity of at least 5",
    "high_intensity_review_any": "Flag for high harvesting intensity in any component",
    "intensity_quality_flag": "Summary harvesting-intensity quality classification",
}
for ha_col, kilo_col in UNIT_COLUMNS.items():
    descriptions[kilo_col] = f"{descriptions[ha_col]}, in thousand hectares"

require(set(descriptions) == set(CALCULATOR_COLUMNS), "Data dictionary is incomplete.")
dictionary = pd.DataFrame([
    {"column": col, "description": descriptions[col],
     "unit": ("ha" if col in AREAS else "1000 ha" if col in UNIT_COLUMNS.values()
              else "ratio" if col.startswith("harvesting_intensity_") else "text or flag")}
    for col in CALCULATOR_COLUMNS
])

year_summary.to_csv(QC_DIR / "04_historical_year_coverage.csv", index=False)
spatial_2015_qc.to_csv(QC_DIR / "04_2015_native_grid_replay_QC.csv", index=False)
mirca_summary.to_csv(QC_DIR / "04_MIRCA23_historical_QC.csv", index=False)
numerical_qc.to_csv(QC_DIR / "04_calculator_numerical_QC.csv", index=False)
final_validation.to_csv(QC_DIR / "04_release_candidate_validation.csv", index=False)
dictionary.to_csv(RELEASE_DIR / "04_historical_data_dictionary.csv", index=False)

changes_copy = RELEASE_DIR / "04_MIRCA2000_approved_23_change_register.csv"
if changes_copy.exists():
    require(digest(changes_copy) == digest(SOURCES["mirca_changes"]),
            "An existing change register differs from Notebook 03.")
else:
    shutil.copy2(SOURCES["mirca_changes"], changes_copy)

print("Versioned historical release candidate:", RELEASE_FILE)
print("Optional Parquet copy:", PARQUET_FILE)
print("Release source SHA-256 retained:", digest(RELEASE_FILE) == candidate_sha)

## 11 | Final manifest and release boundary

The scientific checks can pass before the whole publication package is approved. The final manifest records this difference clearly. The canonical historical FINAL is not overwritten, and full clean-start reproduction remains pending until the complete upstream chain has been rerun in a fresh workspace.

In [ ]:
# 11 | Write the final candidate manifest and review notes

require(digest(SOURCES["canonical_final"]) == canonical_sha,
        "Canonical FINAL changed during this notebook.")
require(digest(RELEASE_FILE) == candidate_sha,
        "Versioned release candidate does not match the validated input.")

public_files = [
    RELEASE_FILE, PARQUET_FILE, changes_copy,
    RELEASE_DIR / "04_historical_data_dictionary.csv",
    *sorted(QC_DIR.glob("04_*.csv")),
    *sorted(QC_DIR.glob("04_*.csv.gz")),
    REPLAY_FILE,
]
public_files = sorted(set(public_files), key=lambda p: str(p))
missing_outputs = [str(p) for p in public_files if not p.is_file()]
require(not missing_outputs, f"Required review outputs missing: {missing_outputs}")

file_inventory = pd.DataFrame([
    {"file": str(p.relative_to(RELEASE_DIR)), "sha256": digest(p), "bytes": p.stat().st_size}
    for p in public_files
])

source_inventory.to_csv(QC_DIR / "04_source_SHA256_inventory.csv", index=False)
file_inventory.to_csv(RELEASE_DIR / "04_candidate_file_inventory.csv", index=False)

release_manifest = {
    "workflow": "SPAM to FABLE historical area publication candidate",
    "notebook": "04_SPAM_FABLE_Historical_Finalization_Publication_Candidate_FINAL.ipynb",
    "version": RELEASE_VERSION,
    "scientific_validation_status": "PASS_RELEASE_CANDIDATE_SCIENTIFIC_VALIDATION",
    "notebook_03_audit_status": manifest_03["status"],
    "historical_records": int(len(historical)),
    "calculator_columns": len(CALCULATOR_COLUMNS),
    "historical_years": YEARS,
    "fable_products": 56,
    "approved_MIRCA2000_share_corrections": 23,
    "year_2015_method": (
        "Mapped 2010 and 2020 5 arc-minute grid-product midpoint of "
        "four primary area components, then frozen 2015 country assignment"
    ),
    "year_2015_grid_replay_status": spatial_2015_qc.loc[0, "status"],
    "year_2015_country_reconciliation_status": country_2015_summary.loc[0, "status"],
    "historical_candidate_sha256": candidate_sha,
    "canonical_final_sha256": canonical_sha,
    "canonical_final_overwritten": False,
    "full_clean_start_reproducibility": "PENDING_FRESH_END_TO_END_REBUILD",
    "canonical_release_promotion": "PENDING_REVIEWED_RELEASE_PROMOTION",
    "public_release_approved": False,
    "validated_input_sha256": dict(zip(source_inventory.asset, source_inventory.sha256)),
    "candidate_files_sha256": dict(zip(file_inventory.file, file_inventory.sha256)),
}

readme = f"""# SPAM to FABLE historical dataset: {RELEASE_VERSION}

This folder contains a checked review candidate, not an approved public release.

## Main file

{RELEASE_FILE.name}

The file keeps the exact validated Notebook 03 historical CSV content. It covers 2000, 2005, 2010, 2015 and 2020, with 56 FABLE products across the covered country-year panels.

## Method

Observed years are 2000, 2005, 2010 and 2020. The 2015 mapped area components were interpolated at native 5 arc-minute grid and product resolution, followed by the locked 2015 country geography assignment. Totals and intensity are derived after aggregating areas. Intensities are never capped at 1.

Year 2000 uses the evidence-based FABLE product completion documented in Notebook 03. It includes accepted FAOSTAT and same-year MIRCA2000 magnitude evidence for selected cases. That source-magnitude reconstruction is separate from the 23 approved MIRCA2000 irrigation-share corrections. The share corrections were integrated into the validated year-2000 country-level candidate without changing harvested or physical area totals. The per-record correction register is included.

Some historical 0/0 harvesting intensities are saved as 0 and others as missing. Neither representation is overwritten by this notebook.

## Validation and reproducibility

Notebook 03 completed eight of eight required scientific audit gates. This Notebook 04 checks the native-grid 2015 component replay, country-level 2015 reconciliation and the complete historical Calculator schema and numerical rules.

The frozen 2015 geography crosswalk is an input to this notebook. Generating that crosswalk again from raw boundaries, and clean-start execution of all upstream notebooks, are separate release tasks and are not claimed complete here.

## Release decision

Scientific candidate validation: PASS.
Canonical Calculator FINAL overwrite: No.
Full clean-start reproduction: Pending.
Public release approval: Pending.

The SHA-256 manifests and QC tables document the files used in this candidate.
"""

readme_path = RELEASE_DIR / "README_publication_candidate.md"
manifest_path = RELEASE_DIR / "04_release_candidate_manifest.json"

if readme_path.exists():
    require(readme_path.read_text(encoding="utf-8") == readme,
            "Existing README differs. Change RELEASE_VERSION before rebuilding.")
else:
    readme_path.write_text(readme, encoding="utf-8")

if manifest_path.exists():
    prior_manifest = load_json(manifest_path)
    # A repeated run is acceptable only if the release content is unchanged.
    require(prior_manifest == release_manifest,
            "An existing candidate manifest differs. Use a new RELEASE_VERSION.")
else:
    manifest_path.write_text(
        json.dumps(release_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )

final_summary = pd.DataFrame([{
    "stage": "NOTEBOOK04_FINAL_HISTORICAL_CANDIDATE",
    "scientific_gates_passed": 4,
    "historical_records": len(historical),
    "calculator_columns": len(CALCULATOR_COLUMNS),
    "MIRCA2000_corrections": 23,
    "2015_spatial_mismatches": int(spatial_2015_qc.loc[0, "component_mismatches"]),
    "2015_country_mismatches": len(country_2015_failures),
    "numerical_failures": int(numerical_qc.failures.sum()),
    "status": "VALIDATED_VERSIONED_CANDIDATE_CLEAN_START_PENDING",
    "public_release_approved": False,
}])

print("\nNOTEBOOK 04 | FINAL RESULT")
display(final_summary)
print("Review candidate:", RELEASE_FILE)
print("QC reports:", QC_DIR)
print("Manifest:", manifest_path)
print("Original canonical FINAL is unchanged.")
print("Next release gate: full fresh upstream rebuild, then reviewed promotion.")
con.close()